# Notebook 4 — Quality-aware cVAE architecture

**Goal:** plug the quality score $c \in [0,1]$ (notebook 3) into the Stanford MedVAE architecture,
to build a **quality-aware conditional variational autoencoder (cVAE)**.

**Conditioning principle:**

$$\text{Encoder}: x_{\text{cond}} = \text{cat}\bigl([x,\; c_{\text{spatial}}]\bigr) \xrightarrow{\text{Encoder}} \mu, \sigma \xrightarrow{\text{reparam.}} z$$

$$\text{Decoder}: z_{\text{cond}} = \text{cat}\bigl([\text{post\_quant\_conv}(z),\; c_{\text{spatial}}]\bigr) \xrightarrow{\text{Decoder}} \hat{x}$$

where $c_{\text{spatial}} \in \mathbb{R}^{B \times 1 \times H \times W}$ is the quality score broadcast spatially.

**Why inject at both ends?**
- **Encoder**: makes the latent representation aware of the quality of the compressed image.
- **Decoder**: lets the network *choose* the level of detail to reconstruct according to the declared quality.

**Initialisation:** the new filters are initialised at zero → at the start, $c$ has no effect (stable fine-tuning).

## 0. Configuration

In [ ]:
from pathlib import Path
from pipeline_config import APPROACH, get_score_column, print_approach_summary

# ── Selected approach ──
print_approach_summary()

# --- Chemins ---
NOTEBOOK_DIR  = Path(".").resolve()
PROJECT_ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# Outputs organised by approach
OUTPUT_DIR      = Path(f"outputs_cvae/04_MedVAE_Architecture_Mod/approach_{APPROACH}")
FIGURES_DIR     = OUTPUT_DIR / "figures"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
for _d in [OUTPUT_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

NB03_DIR      = Path("outputs_cvae/03_Inductive_Bias_Generation")
LABELS_CSV    = NB03_DIR / "labels_quality.csv"  # sortie Notebook 3

# Official pre-trained MedVAE weights (Stanford, medvae_4_4_2d)
# Downloaded once, then reused from the local disk
PRETRAINED_DIR = Path("outputs_cvae/pretrained_weights")
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
PHASE1_CKPT = PRETRAINED_DIR / "vae_4x_4c_2D.ckpt"

if not PHASE1_CKPT.exists():
    import shutil
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id="stanfordmimi/MedVAE", filename="model_weights/vae_4x_4c_2D.ckpt")
    shutil.copy(cached, PHASE1_CKPT)
    print(f"⬇️  Weights downloaded and copied → {PHASE1_CKPT}")
else:
    print(f"✅ Local weights found: {PHASE1_CKPT}")

CVAE_CKPT_OUT = CHECKPOINTS_DIR / "quality_aware_vae.pt"

# --- Model config (must match pretraining.yaml) ---
DDCONFIG = {
    "double_z":        True,
    "z_channels":      4,
    "resolution":      64,
    "in_channels":     1,
    "out_ch":          1,
    "ch":              128,
    "ch_mult":         [1, 2, 4],
    "num_res_blocks":  2,
    "attn_resolutions":[],
    "dropout":         0.0,
}
EMBED_DIM = 4

# --- Training config ---
# Phase 1: FiLM-only warm-up (base VAE frozen)
N_WARMUP_STEPS  = 2_500   # ~10 epochs (1000 images / batch 4 ≈ 250 steps/epoch)
WARMUP_LR       = 4.5e-6

# Phase 2: global fine-tuning (everything unfrozen, lower LR)
N_FINETUNE_STEPS = 5_000  # ~20 more epochs
FINETUNE_LR      = 1e-6   # lower LR so as not to break the pre-trained weights

BATCH_SIZE     = 4
KL_WEIGHT      = 1e-6
VAL_EVERY      = 250      # validation every 250 steps (~1 epoch)
PATIENCE       = 10       # early stopping: stop after 10 validations without improvement

print(f"Labels CSV   : {LABELS_CSV}")
print(f"Checkpoint   : {CVAE_CKPT_OUT}")
print(f"Warm-up      : {N_WARMUP_STEPS} steps  |  Fine-tune : {N_FINETUNE_STEPS} steps")
print(f"Validation   : every {VAL_EVERY} steps  |  Patience: {PATIENCE}")

## 1. Imports et setup sys.path

In [ ]:
import sys
import copy
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
torch.manual_seed(42)

# Import the standalone MedVAE (no need for jepa-adaptation any more)
from medvae_standalone import AutoencoderKL as AutoencoderKL2D
from medvae_standalone import DiagonalGaussianDistribution

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

## 2. Loading labels_quality.csv

In [ ]:
from pipeline_config import read_csv_with_paths
df_labels = read_csv_with_paths(LABELS_CSV)
print(f"Labels loaded: {len(df_labels)} images")

# ⚠ No leakage: check that the split column exists
if "split" in df_labels.columns:
    print(f"Split : {df_labels['split'].value_counts().to_dict()}")
else:
    # Fallback: infer the split from the path
    df_labels["split"] = df_labels["path"].apply(
        lambda p: "train" if "seg_train" in str(p) else ("val" if "seg_val" in str(p) else "unknown")
    )
    print(f"Inferred split: {df_labels['split'].value_counts().to_dict()}")

# Recompute quality_score/quality_label from the column of the
# active approach (instead of trusting the generic aliases of the CSV, which only reflect
# the last approach NB03 was run with): makes this notebook
# robust to the order in which A/B/C are run.
SCORE_COL = get_score_column()
if SCORE_COL not in df_labels.columns:
    raise ValueError(
        f"Column '{SCORE_COL}' missing from {LABELS_CSV} - rerun NB03 with "
        f"APPROACH='{APPROACH}' before this notebook."
    )
df_labels["quality_score"] = df_labels[SCORE_COL]
train_scores = df_labels.loc[df_labels["split"] == "train", SCORE_COL]
t33, t67 = train_scores.quantile(1 / 3), train_scores.quantile(2 / 3)
df_labels["quality_label"] = df_labels[SCORE_COL].apply(
    lambda s: "bad" if s < t33 else ("medium" if s < t67 else "good")
)

print(df_labels[["image_id", "quality_score", "quality_label"]].describe())
df_labels.head(3)

## 3. Dataset: QualityAwareArcadeDataset

A minimal change of the existing DataLoader: it returns `(image, quality_score)` instead of `(image, mask)`.

In [ ]:
class QualityAwareArcadeDataset(Dataset):
    """
    Load the ARCADE images and their quality score.
    
    Returns: (image: FloatTensor (1,H,W), quality_score: scalar FloatTensor)
    """
    def __init__(self, df: pd.DataFrame, img_size: int = 64):
        # Keep only the rows with an accessible image file
        self.records = df.dropna(subset=["path", "quality_score"]).reset_index(drop=True)
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        row = self.records.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("L")
            img = img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
            img = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).unsqueeze(0)
            # Recadrage [-1, 1] (convention MedVAE)
            img = img * 2.0 - 1.0
        except Exception:
            img = torch.zeros(1, self.img_size, self.img_size)

        c = torch.tensor(float(row["quality_score"]), dtype=torch.float32)
        return img, c


# ⚠ No leakage: train on the TRAIN images only
df_train = df_labels[df_labels["split"] == "train"]
df_val   = df_labels[df_labels["split"] == "val"]

dataset     = QualityAwareArcadeDataset(df_train, img_size=DDCONFIG["resolution"])
dataset_val = QualityAwareArcadeDataset(df_val,   img_size=DDCONFIG["resolution"])
loader      = DataLoader(dataset,     batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
loader_val  = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

imgs_demo, scores_demo = next(iter(loader))
print(f"Train : {len(dataset)} images  |  Val : {len(dataset_val)} images")
print(f"Batch images : {imgs_demo.shape}   scores : {scores_demo.shape}")
print(f"Batch scores: {scores_demo.tolist()}")

## 4. Architecture: QualityAwareAutoencoderKL

### 4a. FiLM conditioning

Instead of changing the input convolutions (patching `conv_in`), the quality score $c$ is injected with
**feature-wise linear modulation (FiLM)**:

| Aspect | Concatenation (former) | FiLM (current) |
|---|---|---|
| Injection point | Input only | Every ResnetBlock (19 blocks) |
| Model change | Patch `conv_in` (+1 channel) | None (forward hooks) |
| Modulation | Constant spatial channel | Learned scale + shift per layer |
| Initialisation | Zero (no initial effect) | Zero (identity) |
| Added parameters | ~0.01% | ~2.7% |

**How it works:** a shared MLP maps $c$ to a 128-dimensional embedding.
Then 19 linear heads (one per `ResnetBlock`) produce the parameters $(\gamma_i, \beta_i)$.
After each `ResnetBlock`: $h \leftarrow (1 + \gamma_i) \cdot h + \beta_i$.

In [ ]:
class FiLMConditioner(nn.Module):
    """
    Feature-wise Linear Modulation (FiLM) — Perez et al., AAAI 2018.

    Map a scalar quality score c ∈ [0,1] to affine parameters
    (γ, β) for each ResnetBlock:
        h ← (1 + γ(c)) · h + β(c)

    Zero init ⟹ at the start γ=0, β=0 ⟹ identity (no perturbation).
    """

    def __init__(self, channels_per_block: list, hidden_dim: int = 128):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.heads = nn.ModuleList()
        for ch in channels_per_block:
            head = nn.Linear(hidden_dim, 2 * ch)
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)
            self.heads.append(head)

    def forward(self, c: torch.Tensor) -> list:
        """c: (B,) → list of (γ, β) of shape (B, ch, 1, 1)."""
        emb = self.embed(c.view(-1, 1))
        params = []
        for head in self.heads:
            gb = head(emb)
            gamma, beta = gb.chunk(2, dim=-1)
            params.append((
                gamma.unsqueeze(-1).unsqueeze(-1),
                beta.unsqueeze(-1).unsqueeze(-1),
            ))
        return params


print("FiLMConditioner defined.")

### 4b. QualityAwareAutoencoderKL class (FiLM conditioning)

The quality score $c \in [0, 1]$ is injected through **feature-wise linear modulation (FiLM)**:

1. A shared MLP maps $c$ to a 128-dimensional embedding
2. 19 linear heads (one per ResnetBlock) produce $(\gamma_i, \beta_i)$
3. After each ResnetBlock: $h \leftarrow (1 + \gamma_i) \cdot h + \beta_i$
4. The heads are **zero-initialised** → identity at the start → stability

In [ ]:
class QualityAwareAutoencoderKL(nn.Module):
    """
    cVAE conditioned through FiLM (feature-wise linear modulation).

    Instead of a simple channel concatenation at the input, the score c ∈ [0,1]
    is injected into EVERY residual block of the encoder and decoder through an
    modulation affine apprise :

        h ← (1 + γ(c)) · h + β(c)     after each ResnetBlock

    Advantages over concatenation:
    - Multi-layer modulation (19 blocks) instead of the input only
    - No change of the dimensions of the base convolutions
    - More expressive (scale + shift learned per layer)
    - The base weights stay strictly intact (no conv_in patch)

    Ref: Perez et al., "FiLM: Visual Reasoning with a General Conditioning
    Layer", AAAI 2018.
    """

    def __init__(self, base_ae: AutoencoderKL2D):
        super().__init__()
        self.ae = base_ae

        # List the ResnetBlocks in execution order
        self._blocks = list(self._iter_resnet_blocks())
        channels = [b.conv2.out_channels for b in self._blocks]

        # FiLM generator (zero init → identity at the start)
        self.film = FiLMConditioner(channels)

    def _iter_resnet_blocks(self):
        """Yield the ResnetBlocks in execution order (enc → dec)."""
        enc = self.ae.encoder
        for i_level in range(enc.num_resolutions):
            for i_block in range(enc.num_res_blocks):
                yield enc.down[i_level].block[i_block]
        yield enc.mid.block_1
        yield enc.mid.block_2

        dec = self.ae.decoder
        yield dec.mid.block_1
        yield dec.mid.block_2
        for i_level in reversed(range(dec.num_resolutions)):
            for i_block in range(dec.num_res_blocks + 1):
                yield dec.up[i_level].block[i_block]

    def _register_film_hooks(self, c: torch.Tensor):
        """Register FiLM forward hooks on each ResnetBlock."""
        film_params = self.film(c)
        hooks = []
        for i, block in enumerate(self._blocks):
            gamma, beta = film_params[i]
            hook = block.register_forward_hook(
                lambda mod, inp, out, g=gamma, b=beta: (1 + g) * out + b
            )
            hooks.append(hook)
        return hooks

    def forward(self, x, c, sample_posterior=True):
        hooks = self._register_film_hooks(c)
        try:
            h = self.ae.encoder(x)
            moments = self.ae.quant_conv(h)
            posterior = DiagonalGaussianDistribution(moments)
            z = posterior.sample() if sample_posterior else posterior.mode()
            z_post = self.ae.post_quant_conv(z)
            dec = self.ae.decoder(z_post)
        finally:
            for hk in hooks:
                hk.remove()
        return dec, posterior

    def get_last_layer(self):
        return self.ae.get_last_layer()

    def param_count(self) -> dict:
        total = sum(p.numel() for p in self.parameters())
        film  = sum(p.numel() for p in self.film.parameters())
        return {"total": total, "film_params": film, "ratio_%": round(100 * film / total, 3)}


print("QualityAwareAutoencoderKL defined (FiLM conditioning).")

## 5. Building and checking the architecture

In [ ]:
# Base model
base_ae = AutoencoderKL2D(
    ddconfig         = DDCONFIG,
    embed_dim        = EMBED_DIM,
    ckpt_path        = PHASE1_CKPT,
    apply_channel_ds = True,
)

# ── Snapshot of the pre-trained weights BEFORE any fine-tuning ──
# Serves as the real baseline in notebook 5
pretrained_state_dict = {k: v.cpu().clone() for k, v in base_ae.state_dict().items()}
print(f"Pre-trained snapshot saved: {len(pretrained_state_dict)} tensors")

# Conditioned model (FiLM)
cvae = QualityAwareAutoencoderKL(base_ae).to(DEVICE)

# Summary
pc = cvae.param_count()
print(f"\nTotal parameters     : {pc['total']:,}")
print(f"FiLM parameters      : {pc['film_params']:,}  ({pc['ratio_%']:.3f}% of the total)")
print(f"Number of conditioned ResnetBlocks: {len(cvae._blocks)}")
print(f"\nChannels per block: {[b.conv2.out_channels for b in cvae._blocks]}")

In [ ]:
# --- Check forward pass ---
x_test = imgs_demo.to(DEVICE)
c_test = scores_demo.to(DEVICE)

with torch.no_grad():
    rec, posterior = cvae(x_test, c_test, sample_posterior=False)

print("Forward pass OK :")
print(f"  Input       : {x_test.shape}")
print(f"  Scores c    : {c_test.tolist()}")
print(f"  Reconstruction : {rec.shape}")
print(f"  Latent z    : {posterior.mean.shape}")
print(f"  MSE(x, rec) : {F.mse_loss(x_test, rec).item():.6f}")

In [ ]:
# --- Gradient flow check ---
cvae.train()
x_grad = imgs_demo[:1].to(DEVICE).requires_grad_(False)
c_grad = scores_demo[:1].to(DEVICE).requires_grad_(True)

rec_grad, post_grad = cvae(x_grad, c_grad, sample_posterior=False)
loss_grad = F.mse_loss(rec_grad, x_grad)
loss_grad.backward()

print("Flux de gradient :")
print(f"  ∂loss/∂c              : {c_grad.grad.item():.6f}  (non-nul → c influence la loss)")
print(f"  film.embed[0].w grad  : {cvae.film.embed[0].weight.grad.abs().mean().item():.6f}")
print(f"  film.heads[0].w grad  : {cvae.film.heads[0].weight.grad.abs().mean().item():.6f}")
print(f"  film.heads[-1].w grad : {cvae.film.heads[-1].weight.grad.abs().mean().item():.6f}")
cvae.zero_grad()

## 6. Architecture diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6.5)
ax.axis("off")

def box(ax, x, y, w, h, label, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor="black", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center",
            fontsize=fontsize, fontweight="bold", wrap=True)

def arrow(ax, x1, y, x2, label="", color="black"):
    ax.annotate("", xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
    if label:
        ax.text((x1+x2)/2, y + 0.15, label, ha="center", fontsize=7.5, color=color)

# ── Blocs principaux ──
box(ax, 0.3, 2.5, 1.4, 1.0, "x\n(1, H, W)",           "#d0e8f7")
box(ax, 2.5, 2.5, 2.0, 1.0, "Encoder\n(8 ResBlocks)",  "#b3d9b3")
box(ax, 5.3, 2.5, 1.6, 1.0, "μ, σ\n→ KL → z",         "#b3d9b3")
box(ax, 7.7, 2.5, 2.2, 1.0, "Decoder\n(11 ResBlocks)", "#d0b3d9")
box(ax, 10.7, 2.5, 1.4, 1.0, "x̂\n(1, H, W)",          "#d0e8f7")

# ── Horizontal arrows ──
arrow(ax, 1.7, 3.0, 2.5)
arrow(ax, 4.5, 3.0, 5.3)
arrow(ax, 6.9, 3.0, 7.7)
arrow(ax, 9.9, 3.0, 10.7)

# ── MLP FiLM (en haut) ──
box(ax, 4.5, 5.0, 3.0, 0.8, "FiLM MLP\nc → (γᵢ, βᵢ)₁₉", "#ffe0b3")

# ── Score c ──
ax.text(6.0, 6.2, "c ∈ [0, 1]", ha="center", fontsize=10,
        color="darkorange", fontweight="bold")
ax.annotate("", xy=(6.0, 5.8), xytext=(6.0, 6.1),
            arrowprops=dict(arrowstyle="->", color="darkorange", lw=2))

# ── FiLM arrows → Encoder and Decoder ──
ax.annotate("", xy=(3.5, 3.5), xytext=(5.0, 5.0),
            arrowprops=dict(arrowstyle="->", color="darkorange", lw=1.8, ls="--"))
ax.annotate("", xy=(8.8, 3.5), xytext=(7.0, 5.0),
            arrowprops=dict(arrowstyle="->", color="darkorange", lw=1.8, ls="--"))

# ── Annotation FiLM ──
ax.text(2.4, 4.3, "h ← (1+γ)·h + β\nafter each\nResnetBlock",
        ha="center", fontsize=7.5, color="darkorange", style="italic",
        bbox=dict(facecolor="white", edgecolor="darkorange", alpha=0.8, boxstyle="round,pad=0.2"))

ax.text(9.7, 4.3, "h ← (1+γ)·h + β\nafter each\nResnetBlock",
        ha="center", fontsize=7.5, color="darkorange", style="italic",
        bbox=dict(facecolor="white", edgecolor="darkorange", alpha=0.8, boxstyle="round,pad=0.2"))

# ── Legend ──
legend_handles = [
    mpatches.Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in [
        ("#d0e8f7", "Input/Output"),
        ("#ffe0b3", "FiLM (conditioning)"),
        ("#b3d9b3", "Encodeur"),
        ("#d0b3d9", "Decoder"),
    ]
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8)

ax.set_title("Architecture Quality-Aware VAE — FiLM conditioning multi-couche",
             fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_architecture_cvae.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Visualising the conditioning mechanism

### 7a. Reconstruction at different c on the same image

> **Note:** at initialisation, the FiLM heads are zero-initialised: $\gamma = 0, \beta = 0$,
> so the modulation is the identity $(1+0) \cdot h + 0 = h$. The score $c$ has no visible effect yet.
> After training (section 8), the network learns to modulate differently according to $c$.

In [ ]:
cvae.eval()
x_single = imgs_demo[:1].to(DEVICE)  # a single image

c_values = [0.0, 0.25, 0.5, 0.75, 1.0]
recs = []
with torch.no_grad():
    for c_val in c_values:
        c_t = torch.tensor([c_val], dtype=torch.float32, device=DEVICE)
        rec, _ = cvae(x_single, c_t, sample_posterior=False)
        recs.append(rec.squeeze().cpu().numpy())

fig, axes = plt.subplots(1, len(c_values) + 1, figsize=(3 * (len(c_values) + 1), 3.5))

orig = x_single.squeeze().cpu().numpy()
axes[0].imshow((orig + 1) / 2, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original", fontsize=9)
axes[0].axis("off")

for i, (c_val, rec) in enumerate(zip(c_values, recs)):
    axes[i + 1].imshow((rec + 1) / 2, cmap="gray", vmin=0, vmax=1)
    mse = float(np.mean((orig - rec) ** 2))
    axes[i + 1].set_title(f"c = {c_val:.2f}\nMSE={mse:.5f}", fontsize=8)
    axes[i + 1].axis("off")

fig.suptitle("Reconstruction at different c (before training → FiLM zero init → no effect)",
             fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_conditioning_before_training.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ The MSEs are identical: the FiLM heads are zero-initialised (identity).")

## 8. Training — phase 1: FiLM warm-up

**Two-phase strategy:**

| Phase | Trained parameters | Steps | LR | Goal |
|---|---|---|---|---|
| **1 — Warm-up** | FiLM only (2.7%) | 2,500 | 4.5e-6 | Initialise the conditioning |
| **2 — Fine-tuning** | Whole network | 5,000 | 1e-6 | Adjust the full VAE |

Validation every 250 steps with early stopping (patience = 10).

In [ ]:
def elbo_loss(x, rec, posterior, kl_weight=KL_WEIGHT):
    rec_loss = F.mse_loss(rec, x)
    kl_loss  = posterior.kl().mean()
    loss     = rec_loss + kl_weight * kl_loss
    return loss, {"rec": rec_loss.item(), "kl": kl_loss.item(), "total": loss.item()}


@torch.no_grad()
def validate(model, val_loader, device):
    """Compute the mean loss on the validation set."""
    model.eval()
    total_loss, total_rec, total_kl, n = 0, 0, 0, 0
    for x_batch, c_batch in val_loader:
        x_batch, c_batch = x_batch.to(device), c_batch.to(device)
        rec_batch, post_batch = model(x_batch, c_batch, sample_posterior=False)
        _, logs = elbo_loss(x_batch, rec_batch, post_batch)
        bs = x_batch.size(0)
        total_loss += logs["total"] * bs
        total_rec  += logs["rec"] * bs
        total_kl   += logs["kl"] * bs
        n += bs
    model.train()
    return {"total": total_loss / n, "rec": total_rec / n, "kl": total_kl / n}


print("ELBO loss and validation function defined.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Phase 1: warm-up — only the FiLM parameters are trained
# (the base MedVAE weights are frozen).
# ──────────────────────────────────────────────────────────────

# 1) Freeze all base weights
for p in cvae.parameters():
    p.requires_grad_(False)

# 2) Unfreeze the FiLM parameters only
for p in cvae.film.parameters():
    p.requires_grad_(True)

trainable = [p for p in cvae.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable)
n_total     = sum(p.numel() for p in cvae.parameters())
print(f"Trainable parameters: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.3f}%)")

optimizer = torch.optim.AdamW(trainable, lr=WARMUP_LR, weight_decay=0.0)
cvae.train()

history_warmup = {"total": [], "rec": [], "kl": [], "val_total": [], "val_rec": []}
best_val_loss  = float("inf")
best_state     = None
patience_ctr   = 0
step = 0
loader_iter = iter(loader)

pbar = tqdm(total=N_WARMUP_STEPS, desc="Phase 1 — Warm-up FiLM")

while step < N_WARMUP_STEPS:
    try:
        x_batch, c_batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(loader)
        x_batch, c_batch = next(loader_iter)

    x_batch = x_batch.to(DEVICE)
    c_batch = c_batch.to(DEVICE)

    rec_batch, post_batch = cvae(x_batch, c_batch, sample_posterior=True)
    loss, logs = elbo_loss(x_batch, rec_batch, post_batch)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    optimizer.step()

    for k in ["total", "rec", "kl"]:
        history_warmup[k].append(logs[k])

    step += 1

    # --- Periodic validation ---
    if step % VAL_EVERY == 0 or step == N_WARMUP_STEPS:
        val_logs = validate(cvae, loader_val, DEVICE)
        history_warmup["val_total"].append(val_logs["total"])
        history_warmup["val_rec"].append(val_logs["rec"])
        cvae.train()

        improved = val_logs["total"] < best_val_loss
        if improved:
            best_val_loss = val_logs["total"]
            best_state = {k: v.cpu().clone() for k, v in cvae.state_dict().items()}
            patience_ctr = 0
            marker = " ★"
        else:
            patience_ctr += 1
            marker = ""

        pbar.set_postfix({
            "train": f"{logs['total']:.5f}",
            "val": f"{val_logs['total']:.5f}{marker}",
            "pat": f"{patience_ctr}/{PATIENCE}",
        })

        if patience_ctr >= PATIENCE:
            print(f"\n⚠ Early stopping at step {step} (patience exhausted)")
            break
    else:
        pbar.set_postfix({"loss": f"{logs['total']:.5f}", "rec": f"{logs['rec']:.5f}"})

    pbar.update(1)

pbar.close()

# Restore the best model
if best_state is not None:
    cvae.load_state_dict(best_state)
    print(f"Best model restored (val_loss = {best_val_loss:.6f})")

print(f"Phase 1 done — {step} steps, best val loss: {best_val_loss:.6f}")

In [ ]:
# Courbes de loss — Phase 1 (Warm-up FiLM)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, key, color in zip(axes, ["total", "rec", "kl"],
                           ["steelblue", "darkorange", "mediumpurple"]):
    ax.plot(history_warmup[key], color=color, lw=1, alpha=0.6, label="train")
    # Validation (sampled every VAL_EVERY steps)
    if f"val_{key}" in history_warmup and history_warmup.get(f"val_{key}"):
        val_x = list(range(VAL_EVERY, VAL_EVERY * len(history_warmup[f"val_{key}"]) + 1, VAL_EVERY))
        ax.plot(val_x, history_warmup[f"val_{key}"], "o-", color="red", ms=3, lw=1.5, label="val")
    ax.set_title(f"Loss : {key}")
    ax.set_xlabel("step")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Phase 1 — Warm-up FiLM ({len(history_warmup['total'])} steps)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_warmup_curves.png", dpi=100, bbox_inches="tight")
plt.show()

## 9. After training: the score c starts to have an effect

In [ ]:
cvae.eval()

# Take a "bad" image and a "good" image
df_bad  = df_labels[df_labels["quality_label"] == "bad"].head(1)
df_good = df_labels[df_labels["quality_label"] == "good"].head(1)

def load_and_prepare(row, img_size=DDCONFIG["resolution"]):
    img = Image.open(row["path"].values[0]).convert("L")
    img = img.resize((img_size, img_size), Image.Resampling.BILINEAR)
    t   = torch.from_numpy(np.asarray(img, np.float32) / 255.0).unsqueeze(0).unsqueeze(0)
    return (t * 2.0 - 1.0).to(DEVICE)

imgs_pair  = [load_and_prepare(df_bad),  load_and_prepare(df_good)]
labels_pair= ["Degraded image (bad)",    "Clean image (good)"]
c_real_pair= [float(df_bad["quality_score"].values[0]),
              float(df_good["quality_score"].values[0])]

c_test_vals = [0.0, 0.5, 1.0]

fig, axes = plt.subplots(2, len(c_test_vals) + 2, figsize=(3 * (len(c_test_vals) + 2), 7))

for row, (x_img, lbl, c_real) in enumerate(zip(imgs_pair, labels_pair, c_real_pair)):
    # Colonne 0 : original
    orig_np = ((x_img.squeeze().cpu().numpy() + 1) / 2).clip(0, 1)
    axes[row, 0].imshow(orig_np, cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{lbl}\nc_real={c_real:.3f}", fontsize=8, color="navy")
    axes[row, 0].axis("off")

    # Column 1: reconstruction with the real c
    with torch.no_grad():
        c_t   = torch.tensor([c_real], device=DEVICE)
        rec_r, _ = cvae(x_img, c_t, sample_posterior=False)
    rec_np = ((rec_r.squeeze().cpu().numpy() + 1) / 2).clip(0, 1)
    axes[row, 1].imshow(rec_np, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title(f"Rec c=c_real\nMSE={np.mean((orig_np - rec_np)**2):.5f}",
                            fontsize=8, color="forestgreen")
    axes[row, 1].axis("off")

    # Columns 2+: imposed c
    for col, c_val in enumerate(c_test_vals):
        with torch.no_grad():
            c_t2  = torch.tensor([c_val], device=DEVICE)
            rec_t, _ = cvae(x_img, c_t2, sample_posterior=False)
        rec_np2 = ((rec_t.squeeze().cpu().numpy() + 1) / 2).clip(0, 1)
        axes[row, col + 2].imshow(rec_np2, cmap="gray", vmin=0, vmax=1)
        axes[row, col + 2].set_title(
            f"Rec c={c_val:.1f}\nMSE={np.mean((orig_np - rec_np2)**2):.5f}", fontsize=8
        )
        axes[row, col + 2].axis("off")

fig.suptitle("Effect of the FiLM conditioning after training", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_conditioning_after_training.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Training — phase 2: global fine-tuning

All the weights of the network (base VAE + FiLM) are now unfrozen.
The LR is lowered so as not to destroy the representations learned in pre-training.

In [ ]:
# ──────────────────────────────────────────────────────────────
# Phase 2: global fine-tuning — all weights are trained
# ──────────────────────────────────────────────────────────────

# Unfreeze the whole network
for p in cvae.parameters():
    p.requires_grad_(True)

all_params  = list(cvae.parameters())
n_trainable = sum(p.numel() for p in all_params)
print(f"Trainable parameters: {n_trainable:,} (100%)")

optimizer_ft = torch.optim.AdamW(all_params, lr=FINETUNE_LR, weight_decay=1e-4)
# Cosine scheduler to lower the LR progressively
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ft, T_max=N_FINETUNE_STEPS, eta_min=1e-7
)
cvae.train()

history_ft = {"total": [], "rec": [], "kl": [], "val_total": [], "val_rec": []}
best_val_loss_ft = float("inf")
best_state_ft    = None
patience_ctr_ft  = 0
step_ft = 0
loader_iter_ft = iter(loader)

pbar_ft = tqdm(total=N_FINETUNE_STEPS, desc="Phase 2 — Fine-tuning global")

while step_ft < N_FINETUNE_STEPS:
    try:
        x_batch, c_batch = next(loader_iter_ft)
    except StopIteration:
        loader_iter_ft = iter(loader)
        x_batch, c_batch = next(loader_iter_ft)

    x_batch = x_batch.to(DEVICE)
    c_batch = c_batch.to(DEVICE)

    rec_batch, post_batch = cvae(x_batch, c_batch, sample_posterior=True)
    loss, logs = elbo_loss(x_batch, rec_batch, post_batch)

    optimizer_ft.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_params, 1.0)
    optimizer_ft.step()
    scheduler_ft.step()

    for k in ["total", "rec", "kl"]:
        history_ft[k].append(logs[k])

    step_ft += 1

    # --- Periodic validation ---
    if step_ft % VAL_EVERY == 0 or step_ft == N_FINETUNE_STEPS:
        val_logs = validate(cvae, loader_val, DEVICE)
        history_ft["val_total"].append(val_logs["total"])
        history_ft["val_rec"].append(val_logs["rec"])
        cvae.train()

        improved = val_logs["total"] < best_val_loss_ft
        if improved:
            best_val_loss_ft = val_logs["total"]
            best_state_ft = {k: v.cpu().clone() for k, v in cvae.state_dict().items()}
            patience_ctr_ft = 0
            marker = " ★"
        else:
            patience_ctr_ft += 1
            marker = ""

        lr_now = scheduler_ft.get_last_lr()[0]
        pbar_ft.set_postfix({
            "train": f"{logs['total']:.5f}",
            "val": f"{val_logs['total']:.5f}{marker}",
            "lr": f"{lr_now:.1e}",
            "pat": f"{patience_ctr_ft}/{PATIENCE}",
        })

        if patience_ctr_ft >= PATIENCE:
            print(f"\n⚠ Early stopping at step {step_ft} (patience exhausted)")
            break
    else:
        pbar_ft.set_postfix({"loss": f"{logs['total']:.5f}", "rec": f"{logs['rec']:.5f}"})

    pbar_ft.update(1)

pbar_ft.close()

# Restore the best model
if best_state_ft is not None:
    cvae.load_state_dict(best_state_ft)
    print(f"Best model restored (val_loss = {best_val_loss_ft:.6f})")

print(f"Phase 2 done — {step_ft} steps, best val loss: {best_val_loss_ft:.6f}")
print(f"Total training: {len(history_warmup['total']) + step_ft} steps")

In [ ]:
# Courbes de loss — Phase 2 (Fine-tuning global)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, key, color in zip(axes, ["total", "rec", "kl"],
                           ["steelblue", "darkorange", "mediumpurple"]):
    ax.plot(history_ft[key], color=color, lw=1, alpha=0.6, label="train")
    if f"val_{key}" in history_ft and history_ft.get(f"val_{key}"):
        val_x = list(range(VAL_EVERY, VAL_EVERY * len(history_ft[f"val_{key}"]) + 1, VAL_EVERY))
        ax.plot(val_x, history_ft[f"val_{key}"], "o-", color="red", ms=3, lw=1.5, label="val")
    ax.set_title(f"Loss : {key}")
    ax.set_xlabel("step")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Phase 2 — Fine-tuning global ({len(history_ft['total'])} steps)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_finetune_curves.png", dpi=100, bbox_inches="tight")
plt.show()

## 10b. Training the baseline — vanilla VAE (ablation)

For a rigorous comparison in NB05, an **identical VAE without FiLM** is trained
on the **same data**, with the **same hyperparameters** (phase 2 only).

This isolates the contribution of the conditioning $c$:
- **Baseline** = MedVAE fine-tuned on ARCADE (same compute budget, no FiLM)
- **cVAE** = MedVAE fine-tuned on ARCADE + FiLM conditioning on $c$

In [ ]:
# ──────────────────────────────────────────────────────────────
# Baseline ablation: fine-tuning the vanilla VAE (no FiLM)
# Same data, same total number of steps, same LR as phase 2.
# ──────────────────────────────────────────────────────────────

# Build a new VAE from the pre-trained weights
baseline_vae = AutoencoderKL2D(
    ddconfig=DDCONFIG, embed_dim=EMBED_DIM, apply_channel_ds=True,
).to(DEVICE)
baseline_vae.load_state_dict(pretrained_state_dict)
print(f"Baseline VAE built: {sum(p.numel() for p in baseline_vae.parameters()):,} parameters")

# Same compute budget as the cVAE:
# Total steps = N_WARMUP_STEPS + N_FINETUNE_STEPS
N_BASELINE_STEPS = N_WARMUP_STEPS + N_FINETUNE_STEPS

# Same hyperparameters as phase 2 of the cVAE
all_params_bl = list(baseline_vae.parameters())
for p in all_params_bl:
    p.requires_grad_(True)

optimizer_bl = torch.optim.AdamW(all_params_bl, lr=FINETUNE_LR, weight_decay=1e-4)
scheduler_bl = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_bl, T_max=N_BASELINE_STEPS, eta_min=1e-7
)
baseline_vae.train()

history_bl = {"total": [], "rec": [], "kl": [], "val_total": [], "val_rec": []}
best_val_loss_bl = float("inf")
best_state_bl    = None
patience_ctr_bl  = 0
step_bl = 0
loader_iter_bl = iter(loader)

@torch.no_grad()
def validate_baseline(model, val_loader, device):
    """Validation of the vanilla VAE (no conditioning c)."""
    model.eval()
    total_loss, total_rec, total_kl, n = 0, 0, 0, 0
    for x_batch, _ in val_loader:  # on ignore c_batch
        x_batch = x_batch.to(device)
        rec_batch, posterior_batch, _ = model(x_batch, sample_posterior=False, decode=True)
        _, logs = elbo_loss(x_batch, rec_batch, posterior_batch)
        bs = x_batch.size(0)
        total_loss += logs["total"] * bs
        total_rec  += logs["rec"] * bs
        total_kl   += logs["kl"] * bs
        n += bs
    model.train()
    return {"total": total_loss / n, "rec": total_rec / n, "kl": total_kl / n}


pbar_bl = tqdm(total=N_BASELINE_STEPS, desc="Ablation — baseline VAE (no FiLM)")

while step_bl < N_BASELINE_STEPS:
    try:
        x_batch, _ = next(loader_iter_bl)  # on ignore c_batch
    except StopIteration:
        loader_iter_bl = iter(loader)
        x_batch, _ = next(loader_iter_bl)

    x_batch = x_batch.to(DEVICE)

    # Vanilla VAE forward pass (no conditioning)
    rec_batch, posterior_batch, _ = baseline_vae(x_batch, sample_posterior=True, decode=True)
    loss, logs = elbo_loss(x_batch, rec_batch, posterior_batch)

    optimizer_bl.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(all_params_bl, 1.0)
    optimizer_bl.step()
    scheduler_bl.step()

    for k in ["total", "rec", "kl"]:
        history_bl[k].append(logs[k])

    step_bl += 1

    # --- Periodic validation ---
    if step_bl % VAL_EVERY == 0 or step_bl == N_BASELINE_STEPS:
        val_logs = validate_baseline(baseline_vae, loader_val, DEVICE)
        history_bl["val_total"].append(val_logs["total"])
        history_bl["val_rec"].append(val_logs["rec"])
        baseline_vae.train()

        improved = val_logs["total"] < best_val_loss_bl
        if improved:
            best_val_loss_bl = val_logs["total"]
            best_state_bl = {k: v.cpu().clone() for k, v in baseline_vae.state_dict().items()}
            patience_ctr_bl = 0
            marker = " ★"
        else:
            patience_ctr_bl += 1
            marker = ""

        lr_now = scheduler_bl.get_last_lr()[0]
        pbar_bl.set_postfix({
            "train": f"{logs['total']:.5f}",
            "val": f"{val_logs['total']:.5f}{marker}",
            "lr": f"{lr_now:.1e}",
            "pat": f"{patience_ctr_bl}/{PATIENCE}",
        })

        if patience_ctr_bl >= PATIENCE:
            print(f"\n⚠ Early stopping at step {step_bl} (patience exhausted)")
            break
    else:
        pbar_bl.set_postfix({"loss": f"{logs['total']:.5f}", "rec": f"{logs['rec']:.5f}"})

    pbar_bl.update(1)

pbar_bl.close()

# Restore the best model
if best_state_bl is not None:
    baseline_vae.load_state_dict(best_state_bl)
    print(f"Best baseline restored (val_loss = {best_val_loss_bl:.6f})")

print(f"Baseline done — {step_bl} steps, best val loss: {best_val_loss_bl:.6f}")

In [ ]:
# Courbes de loss — Baseline VAE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, key, color in zip(axes, ["total", "rec", "kl"],
                           ["steelblue", "darkorange", "mediumpurple"]):
    ax.plot(history_bl[key], color=color, lw=1, alpha=0.6, label="train")
    if f"val_{key}" in history_bl and history_bl.get(f"val_{key}"):
        val_x = list(range(VAL_EVERY, VAL_EVERY * len(history_bl[f"val_{key}"]) + 1, VAL_EVERY))
        ax.plot(val_x, history_bl[f"val_{key}"], "o-", color="red", ms=3, lw=1.5, label="val")
    ax.set_title(f"Loss : {key}")
    ax.set_xlabel("step")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Baseline ablation — VAE without FiLM ({len(history_bl['total'])} steps)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_baseline_curves.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
total_steps = len(history_warmup["total"]) + len(history_ft["total"])

checkpoint = {
    "model_state_dict":       cvae.state_dict(),
    "pretrained_state_dict":  pretrained_state_dict,   # ← poids MedVAE AVANT fine-tuning
    "baseline_state_dict":    baseline_vae.state_dict(),  # ← VAE fine-tuned WITHOUT FiLM (ablation)
    "ddconfig":               DDCONFIG,
    "embed_dim":              EMBED_DIM,
    "approach":               APPROACH,
    "training_steps":         total_steps,
    "baseline_steps":         len(history_bl["total"]),
    "warmup_steps":           len(history_warmup["total"]),
    "finetune_steps":         len(history_ft["total"]),
    "best_val_loss":          best_val_loss_ft,
    "best_val_loss_baseline": best_val_loss_bl,
    "history_warmup":         history_warmup,
    "history_finetune":       history_ft,
    "history_baseline":       history_bl,
    "labels_csv":             str(LABELS_CSV),
    "param_count":            cvae.param_count(),
}

torch.save(checkpoint, CVAE_CKPT_OUT)
print(f"Checkpoint saved → {CVAE_CKPT_OUT}")
print(f"Approach         : {APPROACH}")
print(f"File size        : {CVAE_CKPT_OUT.stat().st_size / 1e6:.1f} MB")
print(f"Steps cVAE       : {total_steps}  (warmup {len(history_warmup['total'])} + ft {len(history_ft['total'])})")
print(f"Steps baseline   : {len(history_bl['total'])}")
print(f"Best val loss cVAE    : {best_val_loss_ft:.6f}")
print(f"Best val loss baseline: {best_val_loss_bl:.6f}")
print(f"Contains baseline_state_dict: ✓ (ablation for NB05)")

In [ ]:
# Reload check
def load_quality_aware_vae(ckpt_path: Path) -> QualityAwareAutoencoderKL:
    ckpt    = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    base_ae = AutoencoderKL2D(
        ddconfig         = ckpt["ddconfig"],
        embed_dim        = ckpt["embed_dim"],
        apply_channel_ds = True,
    )
    model = QualityAwareAutoencoderKL(base_ae)
    model.load_state_dict(ckpt["model_state_dict"])
    return model

cvae_reloaded = load_quality_aware_vae(CVAE_CKPT_OUT)
print(f"Rechargement OK : {cvae_reloaded.param_count()}")
print(f"Training steps  : {checkpoint['training_steps']}")
print(f"Best val loss   : {checkpoint['best_val_loss']:.6f}")

## Summary of notebook 4

| Component | Change | Detail |
|---|---|---|
| **FiLM MLP** | `nn.Linear(1 → 128 → 128)` | Shared embedding of the score $c$ |
| **FiLM heads** | 19 × `nn.Linear(128 → 2·ch)` | $(\gamma_i, \beta_i)$ per ResnetBlock |
| **Modulation** | $h \leftarrow (1 + \gamma) \cdot h + \beta$ | After each ResnetBlock (19 blocks) |
| Existing weights | **Unchanged** | No change of the base model |
| Zero init | All FiLM heads | No effect at the start → stability |
| `QualityAwareArcadeDataset` | Returns `(img, c)` | c read from `labels_quality.csv` |
| FiLM parameters | **~2.7%** of the total | Expressive but light |

### Two-phase training (cVAE)

| Phase | Parameters | Steps | LR | Scheduler |
|---|---|---|---|---|
| **1 — Warm-up** | FiLM only (2.7%) | 2,500 | 4.5e-6 | — |
| **2 — Fine-tuning** | Whole network (100%) | 5,000 | 1e-6 | Cosine → 1e-7 |

### Baseline ablation (VAE without FiLM)

| | Baseline | cVAE |
|---|---|---|
| Architecture | AutoencoderKL2D | AutoencoderKL2D + FiLM |
| Conditioning | None | Score $c$ through FiLM |
| Total steps | 7,500 | 7,500 (2,500 + 5,000) |
| LR | 1e-6 | 4.5e-6 (warm-up) then 1e-6 |
| Scheduler | Cosine → 1e-7 | Cosine → 1e-7 |
| Data | ARCADE (same split) | ARCADE (same split) |

The baseline is the **experimental control** for NB05:
any improvement of the cVAE is attributable to the conditioning, not to the fine-tuning.

### FiLM vs channel concatenation

| | Concatenation | FiLM |
|---|---|---|
| Injection point | Input only | Every layer (19 blocks) |
| Model change | Patch `conv_in` | None (hooks) |
| Modulation type | Extra constant channel | Learned scale + shift |
| Expressiveness | Low (1 spatial signal) | High (multi-scale modulation) |
| Reference | — | Perez et al., AAAI 2018 |

**Next notebook:** `05_Evaluation_of_Robustness.ipynb` —
rigorous ablation: baseline (fine-tuned without c) vs cVAE (fine-tuned with c).